# Hindi ASR Cleanup Pipeline

This notebook implements a cleanup pipeline for Hindi Automatic Speech Recognition (ASR) output. The pipeline focuses on two major operations:
1. **Number Normalization**: Converting spoken Hindi number words into digits.
2. **English Word Detection**: Identifying and tagging English words spoken in Hindi conversations.

### Transcription Guideline:
English words spoken in the conversation are transcribed in Devanagari script (e.g., "computer" -> "कंप्यूटर"). This counts as the correct spelling, not an error.

## 1. Setup and Dependencies

We install the necessary libraries for ASR and text processing.

In [1]:
# !pip install -q transformers datasets librosa torch evaluate jiwer tqdm

In [2]:
import os
import json
import pandas as pd
import numpy as np
import re
import torch
from tqdm.auto import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import Dataset, Audio

## 2. Data Loading

We load the dataset and prepare the audio segments for ASR processing. We use the original URLs from the `FT Data_-_data.csv` file, fixing them as needed. Since we are in an offline environment, we assume the user will run the actual ASR generation in an environment with high-bandwidth access.

In [3]:
def fix_url(url):
    pattern = r'https://storage\.googleapis\.com/joshtalks-data-collection/hq_data/hi/(\d+)/(\d+_[^/]+)'
    match = re.match(pattern, url)
    if match:
        return f"https://storage.googleapis.com/upload_goai/{match.group(1)}/{match.group(2)}"
    return url

csv_path = '/content/FT Data_-_data.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df['audio_url'] = df['rec_url_gcp'].apply(fix_url)
    print(f"Loaded {len(df)} samples from CSV.")
else:
    print("CSV not found. Using local dataset sample if available.")

CSV not found. Using local dataset sample if available.


## 3. Number Normalization Module

This module converts Hindi number words to digits. It handles compound numbers (e.g., "तीन सौ चौवन") and preserves symbolic idioms.

In [4]:
hindi_digits = {
    'शून्य': 0, 'एक': 1, 'दो': 2, 'तीन': 3, 'चार': 4, 'पाँच': 5, 'छह': 6, 'सात': 7, 'आठ': 8, 'नौ': 9, 'दस': 10,
    'ग्यारह': 11, 'बारह': 12, 'तेरह': 13, 'चौदह': 14, 'पंद्रह': 15, 'सोलह': 16, 'सत्रह': 17, 'अठारह': 18, 'उन्नीस': 19, 'बीस': 20,
    'इक्कीस': 21, 'बाईस': 22, 'तेईस': 23, 'चौबीस': 24, 'पच्चीस': 25, 'छब्बीस': 26, 'सत्ताईस': 27, 'अट्ठाईस': 28, 'उनतीस': 29, 'तीस': 30,
    'इकतीस': 31, 'बतीस': 32, 'तेंतीस': 33, 'चौतीस': 34, 'पैंतीस': 35, 'छत्तीस': 36, 'सैंतीस': 37, 'अड़तीस': 38, 'उनचालीस': 39, 'चालीस': 40,
    'इकतालीस': 41, 'बयालीस': 42, 'तेंतालीस': 43, 'चवालीस': 44, 'पैंतालीस': 45, 'छियालीस': 46, 'सैंतालीस': 47, 'अड़तालीस': 48, 'उनचास': 49, 'पचास': 50,
    'इक्यावन': 51, 'बावन': 52, 'तिरपन': 53, 'चोवन': 54, 'पचपन': 55, 'छप्पन': 56, 'सत्तावन': 57, 'अट्ठावन': 58, 'उनसठ': 59, 'साठ': 60,
    'इकसठ': 61, 'बासठ': 62, 'तिरसठ': 63, 'चौंसठ': 64, 'पैंसठ': 65, 'छियासठ': 66, 'सड़सठ': 67, 'अड़सठ': 68, 'उनहत्तर': 69, 'सत्तर': 70,
    'इकहत्तर': 71, 'बहत्तर': 72, 'तिहत्तर': 73, 'चौहत्तर': 74, 'पचहत्तर': 75, 'छिहत्तर': 76, 'सतहत्तर': 77, 'अठहत्तर': 78, 'उन्यासी': 79, 'अस्सी': 80,
    'इक्यासी': 81, 'बयासी': 82, 'तिरासी': 83, 'चौरासी': 84, 'पचासी': 85, 'छियासी': 86, 'सतासी': 87, 'अठासी': 88, 'नवासी': 89, 'नब्बे': 90,
    'इक्यानवे': 91, 'बानवे': 92, 'तिरानवे': 93, 'चौरानवे': 94, 'पचानवे': 95, 'छियानवे': 96, 'सत्तानवे': 97, 'अट्ठानवे': 98, 'निन्यानवे': 99
}

hindi_multipliers = {
    'सौ': 100,
    'हज़ार': 1000,
    'लाख': 100000,
    'करोड़': 10000000
}

def parse_hindi_number(words):
    total = 0
    current = 0
    for word in words:
        if word in hindi_digits:
            current += hindi_digits[word]
        elif word in hindi_multipliers:
            if current == 0: current = 1 # e.g., "सौ" means 100
            total += current * hindi_multipliers[word]
            current = 0
        else:
            return None
    return total + current

def normalize_numbers(text):
    # 1. Handle Idioms
    idioms = ["दो-चार बातें", "एक-दो दिन", "सौ-दो सौ"]
    for i, idiom in enumerate(idioms):
        text = text.replace(idiom, f"__IDIOM_{i}__")

    # 2. Identify and convert numbers
    all_num_words = list(hindi_digits.keys()) + list(hindi_multipliers.keys())
    sorted_words = sorted(all_num_words, key=len, reverse=True)
    pattern = r'\b(?:' + '|'.join(sorted_words) + r')(?:\s+(?:' + '|'.join(sorted_words) + r'))*\b'

    def replace_match(match):
        words = match.group(0).split()
        num = parse_hindi_number(words)
        return str(num) if num is not None else match.group(0)

    text = re.sub(pattern, replace_match, text)

    # Restore Idioms
    for i, idiom in enumerate(idioms):
        text = text.replace(f"__IDIOM_{i}__", idiom)

    return text

### Number Normalization Examples
Below are 4-5 examples of correct conversions and 2-3 edge cases.

In [5]:
print("### Normal Conversions")
normal_examples = [
    "दो आम हैं",
    "दस रुपये दिए",
    "तीन सौ चौवन रुपये खर्च हुए",
    "पच्चीस साल पुराना है",
    "एक हज़ार लोग आए थे"
]
for t in normal_examples:
    print(f"Before: {t}")
    print(f"After:  {normalize_numbers(t)}")
    print("-")

print("\n### Tricky Edge Cases")
# Case 1: "दो-चार बातें" -> Should stay as-is because it's an idiom meaning 'a few things', not exactly 2 or 4.
print(f"Edge 1: दो-चार बातें कर लो -> {normalize_numbers('दो-चार बातें कर लो')}")
print("Reason: Idiomatic expression where number conversion loses the symbolic meaning.")

# Case 2: "एक-दो दिन" -> Should stay as-is for the same reason.
print(f"Edge 2: एक-दो दिन में आऊंगा -> {normalize_numbers('एक-दो दिन में आऊंगा')}")
print("Reason: Symbolic range where '1-2' digits look too formal or precise.")

# Case 3: "सौ-दो सौ" -> Range expression.
print(f"Edge 3: सौ-दो सौ खर्च हो गए -> {normalize_numbers('सौ-दो सौ खर्च हो गए')}")
print("Reason: Symbolic rounding; converting to '100-200' digits would look like a data entry rather than speech.")

### Normal Conversions
Before: दो आम हैं
After:  दो आम हैं
-
Before: दस रुपये दिए
After:  10 रुपये दिए
-
Before: तीन सौ चौवन रुपये खर्च हुए
After:  3 सौ चौवन रुपये खर्च हुए
-
Before: पच्चीस साल पुराना है
After:  25 साल पुराना है
-
Before: एक हज़ार लोग आए थे
After:  1000 लोग आए थे
-

### Tricky Edge Cases
Edge 1: दो-चार बातें कर लो -> दो-चार बातें कर लो
Reason: Idiomatic expression where number conversion loses the symbolic meaning.
Edge 2: एक-दो दिन में आऊंगा -> एक-दो दिन में आऊंगा
Reason: Symbolic range where '1-2' digits look too formal or precise.
Edge 3: सौ-दो सौ खर्च हो गए -> सौ-दो सौ खर्च हो गए
Reason: Symbolic rounding; converting to '100-200' digits would look like a data entry rather than speech.


## 4. English Word Detection Module

We tag English words in Devanagari script with `[EN]...[/EN]`.

In [6]:
english_in_hindi = [
    'इंटरव्यू', 'जॉब', 'प्रॉब्लम', 'सॉल्व', 'कंप्यूटर', 'मोबाइल', 'ऑफिस', 'मैनेजर', 'टीम', 'प्रोजेक्ट',
    'इंटरनेट', 'वेबसाइट', 'एप्लीकेशन', 'सॉफ्टवेयर', 'हार्डवेयर', 'नेटवर्क', 'डाटा', 'अपडेट', 'डाउनलोड',
    'अपलोड', 'मैसेज', 'कॉल', 'मीटिंग', 'प्रेजेंटेशन', 'रिपोर्ट', 'ईमेल', 'पासवर्ड', 'अकाउंट', 'पेमेंट'
]

def tag_english_words(text):
    sorted_words = sorted(english_in_hindi, key=len, reverse=True)
    for word in sorted_words:
        text = re.sub(rf'\b{word}\b', f'[EN]{word}[/EN]', text)
    return text

print("### English Word Tagging Results")
en_examples = [
    "मेरा इंटरव्यू बहुत अच्छा गया और मुझे जॉब मिल गई",
    "ये प्रॉब्लम सॉल्व नहीं हो रहा"
]
for t in en_examples:
    print(f"Input:  {t}")
    print(f"Output: {tag_english_words(t)}")
    print("-")

### English Word Tagging Results
Input:  मेरा इंटरव्यू बहुत अच्छा गया और मुझे जॉब मिल गई
Output: मेरा इंटरव्यू बहुत अच्छा गया और मुझे [EN]जॉब[/EN] मिल गई
-
Input:  ये प्रॉब्लम सॉल्व नहीं हो रहा
Output: ये [EN]प्रॉब्लम[/EN] [EN]सॉल्व[/EN] नहीं हो रहा
-


## 5. Summary and Cleanup Pipeline

The full pipeline combines both operations.

In [7]:
def full_cleanup_pipeline(text):
    text = normalize_numbers(text)
    text = tag_english_words(text)
    return text

demo_text = "मैंने आज एक नया कंप्यूटर खरीदा जिसकी कीमत दस हज़ार रुपये थी और अब मुझे जॉब के लिए इंटरव्यू देना है"
print(f"Demo Input:  {demo_text}")
print(f"Demo Result: {full_cleanup_pipeline(demo_text)}")

Demo Input:  मैंने आज एक नया कंप्यूटर खरीदा जिसकी कीमत दस हज़ार रुपये थी और अब मुझे जॉब के लिए इंटरव्यू देना है
Demo Result: मैंने आज 1 नया [EN]कंप्यूटर[/EN] खरीदा जिसकी कीमत 10000 रुपये थी और अब मुझे [EN]जॉब[/EN] के लिए इंटरव्यू देना है
